In [ ]:
# 安装所有必要的库
%pip install -qU langchain langchain-openai chromadb tiktoken dashscope

In [3]:
from langchain_core.documents import Document

# 创建一个包含多个主题的示例文档
sample_text = """
人工智能（AI）是计算机科学的一个分支，它企图了解智能的实质，并生产出一种新的能以人类智能相似的方式做出反应的智能机器。该领域的研究包括机器人、语言识别、图像识别、自然语言处理和专家系统等。

AI的历史可以追溯到20世纪50年代。1956年的达特茅斯会议被广泛认为是AI研究的起点。早期的研究集中在解决问题和符号方法上。到了80年代，机器学习开始成为主流。

进入21世纪，深度学习的突破性进展极大地推动了AI的发展。特别是基于Transformer架构的大型语言模型（LLM），如GPT系列，展现了惊人的语言生成和理解能力。这些模型通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

AI的应用已经渗透到各个行业。在医疗领域，AI被用于辅助诊断，通过分析医学影像来识别病变。在金融领域，AI用于风险评估和欺诈检测。在自动驾驶汽车中，AI是实现环境感知和决策控制的核心技术。未来，AI有望在科学研究、教育和娱乐等更多领域带来革命性的变革。
"""

# 将示例文本封装成LangChain的Document对象
docs = [Document(page_content=sample_text, metadata={"source": "ai_report_sample.txt"})]

print(f"示例文档加载完成，总字符数: {len(docs[0].page_content)}")

示例文档加载完成，总字符数: 431


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 定义父切分器 (Parent Splitter)
# 用于创建较大的、上下文完整的父块
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # 设置一个较大的块大小，以保留完整的上下文
    chunk_overlap=200,  # 设置块之间的重叠，以保证连续性
)

# 2. 定义子切分器 (Child Splitter)
# 用于创建较小的、语义集中的子块，适合精准检索
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50  # 设置一个较小的块大小，以生成精准的嵌入
)

print("父子切分器定义完成。")

父子切分器定义完成。


In [6]:
import os

os.environ["OPENAI_API_KEY"] = "sk-****************************"
os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"
os.environ["OPENAI_MODEL"] = "qwen-turbo"
os.environ["EMBEDDING_MODEL"] = "text-embedding-v4"

In [7]:
from langchain_community.embeddings import DashScopeEmbeddings

embeddings = DashScopeEmbeddings(
    model="text-embedding-v4",  # 或 text-embedding-v2
    dashscope_api_key=os.getenv("OPENAI_API_KEY"),
)

In [9]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 初始化向量数据库，用于存储和索引【子块】
# 我们为这个集合命名，以区分不同的实验
vectorstore = Chroma(collection_name="parent_child_demo", embedding_function=embeddings)

print("向量数据库（Chroma）配置完成，用于存储子块。")

向量数据库（Chroma）配置完成，用于存储子块。


In [10]:
from langchain.storage import InMemoryStore

# 初始化文档存储，用于存储【父块】的原文
# 这是一个简单的内存键值存储
store = InMemoryStore()

print("文档存储（InMemoryStore）配置完成，用于存储父块。")

文档存储（InMemoryStore）配置完成，用于存储父块。


In [ ]:
from uuid import uuid4

import chromadb
from langchain_chroma import Chroma

# 导入LangChain相关模块
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
from langchain.retrievers import ParentDocumentRetriever

# 实例化 ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 添加文档到检索器中
# 这一步会自动执行父子分块、嵌入和存储的完整流程
retriever.add_documents(docs, ids=None)

print("ParentDocumentRetriever 实例化并添加文档完成。")

ParentDocumentRetriever 实例化并添加文档完成。


In [12]:
# 检查文档存储中父块的数量
parent_keys = list(store.yield_keys())
print(f"文档存储中有 {len(parent_keys)} 个父块。")

# 检查向量数据库中子块的数量
# 注意：这是一个简化的检查方式，实际子块数量可能更多
# ChromaDB的API不直接暴露条目总数，我们通过一次广泛搜索来估算
child_docs_count = len(vectorstore.similarity_search("AI", k=100))
print(f"向量数据库中至少有 {child_docs_count} 个子块。")

文档存储中有 1 个父块。
向量数据库中至少有 2 个子块。


In [13]:
query = "深度学习对AI有什么影响？"

# 直接在向量数据库中搜索，返回的是【子块】
sub_docs = vectorstore.similarity_search(query, k=2)

print("--- 直接从Vectorstore检索到的子块 ---")
for i, doc in enumerate(sub_docs):
    print(f"子块 {i+1} (长度: {len(doc.page_content)} 字符):")
    print(doc.page_content)
    print("-" * 30)

--- 直接从Vectorstore检索到的子块 ---
子块 1 (长度: 248 字符):
进入21世纪，深度学习的突破性进展极大地推动了AI的发展。特别是基于Transformer架构的大型语言模型（LLM），如GPT系列，展现了惊人的语言生成和理解能力。这些模型通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

AI的应用已经渗透到各个行业。在医疗领域，AI被用于辅助诊断，通过分析医学影像来识别病变。在金融领域，AI用于风险评估和欺诈检测。在自动驾驶汽车中，AI是实现环境感知和决策控制的核心技术。未来，AI有望在科学研究、教育和娱乐等更多领域带来革命性的变革。
------------------------------
子块 2 (长度: 179 字符):
人工智能（AI）是计算机科学的一个分支，它企图了解智能的实质，并生产出一种新的能以人类智能相似的方式做出反应的智能机器。该领域的研究包括机器人、语言识别、图像识别、自然语言处理和专家系统等。

AI的历史可以追溯到20世纪50年代。1956年的达特茅斯会议被广泛认为是AI研究的起点。早期的研究集中在解决问题和符号方法上。到了80年代，机器学习开始成为主流。
------------------------------


In [14]:
# 通过 ParentDocumentRetriever 检索，返回的是【父块】
retrieved_docs = retriever.invoke(query)

print("\n--- 通过ParentDocumentRetriever检索到的父块 ---")
for i, doc in enumerate(retrieved_docs):
    print(f"父块 {i+1} (长度: {len(doc.page_content)} 字符):")
    print(doc.page_content)
    print("-" * 30)


--- 通过ParentDocumentRetriever检索到的父块 ---
父块 1 (长度: 429 字符):
人工智能（AI）是计算机科学的一个分支，它企图了解智能的实质，并生产出一种新的能以人类智能相似的方式做出反应的智能机器。该领域的研究包括机器人、语言识别、图像识别、自然语言处理和专家系统等。

AI的历史可以追溯到20世纪50年代。1956年的达特茅斯会议被广泛认为是AI研究的起点。早期的研究集中在解决问题和符号方法上。到了80年代，机器学习开始成为主流。

进入21世纪，深度学习的突破性进展极大地推动了AI的发展。特别是基于Transformer架构的大型语言模型（LLM），如GPT系列，展现了惊人的语言生成和理解能力。这些模型通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

AI的应用已经渗透到各个行业。在医疗领域，AI被用于辅助诊断，通过分析医学影像来识别病变。在金融领域，AI用于风险评估和欺诈检测。在自动驾驶汽车中，AI是实现环境感知和决策控制的核心技术。未来，AI有望在科学研究、教育和娱乐等更多领域带来革命性的变革。
------------------------------


In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# --- 对照组：传统分块方法 ---
from langchain_openai import ChatOpenAI

# 1. 传统切分器
traditional_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
traditional_chunks = traditional_splitter.split_documents(docs)

# 2. 传统向量存储和检索器
traditional_vectorstore = Chroma.from_documents(
    documents=traditional_chunks,
    embedding=embeddings,
    collection_name="traditional_demo",
)
traditional_retriever = traditional_vectorstore.as_retriever(search_kwargs={"k": 3})

# --- LLM和提示模板 (两组共用) ---
llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,  # 稳定输出
    timeout=1200,  # 超时保护（秒）
    max_retries=2,  # 简单重试
)

template = """
根据以下上下文信息，回答用户的问题。
如果上下文没有提供足够信息，请说明你无法从给定信息中找到答案。

上下文:
{context}

问题: {question}

答案:
"""
prompt = ChatPromptTemplate.from_template(template)

# --- 构建RAG链 ---


def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# 对照组RAG链
traditional_rag_chain = (
    {"context": traditional_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 实验组RAG链 (使用我们之前创建的父子检索器)
parent_child_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# --- 执行查询 ---
query = "请详细解释AI从早期研究到深度学习时代的关键转变及其在各行业的具体应用。"

print("--- 正在执行传统分块方法的RAG查询... ---")
traditional_answer = traditional_rag_chain.invoke(query)
print("\n【传统方法生成的答案】:\n", traditional_answer)

print("\n\n--- 正在执行父子分段方法的RAG查询... ---")
parent_child_answer = parent_child_rag_chain.invoke(query)
print("\n【父子分段方法生成的答案】:\n", parent_child_answer)

--- 正在执行传统分块方法的RAG查询... ---

【传统方法生成的答案】:
 AI从早期研究到深度学习时代的关键转变主要体现在研究方法和技术的演进上。在早期，AI的研究集中在解决问题和符号方法上，即通过逻辑推理和规则来模拟智能行为。到了80年代，机器学习开始成为主流，这标志着AI研究的一个重要转折点，因为机器学习允许计算机通过数据自动学习和改进，而不是依赖于人工设定的规则。

进入21世纪，深度学习的突破性进展极大地推动了AI的发展。特别是基于Transformer架构的大型语言模型（LLM），如GPT系列，展现了惊人的语言生成和理解能力。这些模型通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

在各行业的具体应用方面，AI已经渗透到多个领域。在医疗领域，AI被用于辅助诊断，通过分析医学影像来识别病变。在金融领域，AI用于风险评估和欺诈检测。在自动驾驶汽车中，AI是实现环境感知和决策控制的核心技术。未来，AI有望在科学研究、教育和娱乐等更多领域带来革命性的变革。


--- 正在执行父子分段方法的RAG查询... ---

【父子分段方法生成的答案】:
 AI从早期研究到深度学习时代的关键转变主要体现在以下几个方面：

1. **早期研究（20世纪50年代至80年代）**：
   - AI的起源可以追溯到20世纪50年代，1956年的达特茅斯会议被认为是AI研究的起点。
   - 早期的研究集中在解决问题和符号方法上，即通过逻辑推理和规则来模拟智能行为。
   - 这一阶段的AI系统通常依赖于人工编写的规则和知识库，缺乏自主学习能力。

2. **机器学习的兴起（80年代）**：
   - 到了80年代，机器学习开始成为AI研究的主流方向。
   - 与早期基于规则的方法不同，机器学习允许计算机通过数据自动学习模式和规律，而不需要显式编程。
   - 这一阶段的AI系统能够处理更复杂的任务，并在一定程度上适应新的情况。

3. **深度学习的突破（21世纪）**：
   - 进入21世纪后，深度学习技术的出现极大地推动了AI的发展。
   - 深度学习是一种基于神经网络的机器学习方法，能够自动提取数据的多层次特征。
   - 特别是基于Transformer架构的大型语言模型（如GPT系列），展现了惊人的语言生成和理解能力。这些模型通过在